# NB09 — Finding Structure: Clustering and PCA
**Machine Learning in Finance · Week 9**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB09_clustering_pca.ipynb)

**Finance question:** Which stocks really move together, and how many real bets does a portfolio contain?

**Key idea:** **Unsupervised learning** has no target to predict. **Clustering** (k-means) groups stocks whose returns move together. **PCA** finds the few big common movements behind hundreds of stocks.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## Step 1: who moves with whom?
Monthly returns for every stock, then a correlation heatmap for 40 of them, sorted by official sector.
Red = move together, blue = move in opposite directions.

In [ ]:
# 🟡 YOUR TURN
START, END = "2015", "2025"   # ← the period to learn from; try "2020", "2025"

returns = mlfin.to_returns(mlfin.load_prices("M")).loc[START:END]
_ = mlfin.plot_correlation(returns, n=40)

*What to look for:* blocks of red along the diagonal (stocks in the same sector move together) and a
pale pink everywhere else: almost every stock moves a little with the market.

## Step 2: let the machine find the groups
k-means sorts stocks into K groups so that stocks in the same group have similar up-and-down patterns.
It is never told the sectors. Does it rediscover them?

In [ ]:
# 🟡 YOUR TURN
K = 8   # ← number of clusters: try 4 or 12

labels = mlfin.cluster_stocks(returns, k=K)
info = mlfin.load_universe().set_index("ticker")
sectors = info["sector"].reindex(labels.index)
pd.crosstab(labels, sectors)

In [ ]:
# what is in each cluster? the biggest sector and six example companies
summary = []
for c in sorted(labels.unique()):
    members = labels.index[labels == c]
    top = sectors[members].value_counts()
    summary.append({"cluster": c, "stocks": len(members), "biggest sector": top.index[0],
                    "share of that sector": f"{top.iloc[0] / len(members):.0%}",
                    "examples": ", ".join(info["name"].reindex(members[:6]).fillna("").str[:18])})
pd.DataFrame(summary).set_index("cluster")

*What to look for:* some clusters are almost pure sectors (utilities, real estate, energy), others mix
sectors that react to the same forces (interest rates, oil, consumer spending). Those surprising
groupings are the interesting part.

## Step 3: how many real bets?
PCA finds the biggest **common movement** across stocks, then the next biggest, and so on. If one
component explains a large share, your "30 different stocks" are mostly one bet.

In [ ]:
# 🟡 YOUR TURN
N_STOCKS = 30   # ← portfolio size: try 10 or 100

shares = mlfin.pca_bets(returns, n_stocks=N_STOCKS)

## Step 4: does it help to pick across clusters?
Build many 8-stock portfolios three ways: all from **one** cluster, completely **at random**, and **one
from each** cluster. Which swings least?

In [ ]:
mlfin.diversification_experiment(returns, labels, n=K)

*What to look for:* the big danger is **concentration**. Ten stocks from one cluster are not ten bets.
Random picks from 500 stocks already spread across clusters; picking one per cluster does about as well.

## Step 5: are the course model's picks concentrated?
An ML strategy has no idea about clusters. Check the last 30 stocks the NB07 backtest bought: how many
clusters do they cover?

In [ ]:
bt = mlfin.walk_forward_backtest(mlfin.make_features(), mlfin.default_model(), rebalance="Q", verbose=False)
last_date = max(bt["picks"])
picks = pd.Series(bt["picks"][last_date])
in_cluster = labels.reindex(picks).value_counts().rename("picks in cluster")
print(f"Picks on {last_date:%b %Y}: {picks.notna().sum()} stocks across {in_cluster.size} of {K} clusters")
in_cluster.to_frame()

## ⭐ GOING FURTHER (optional)
1. Use daily returns for a sharper picture: `mlfin.to_returns(mlfin.load_prices("D")).loc["2023":"2025"]`.
   Do the clusters look more like sectors?
2. Re-run Step 2 with `START = "2020"`. Do the clusters change after COVID?
3. Ask an AI assistant: *"What does the first principal component of stock returns represent?"*

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*